# SIGMOD Exp 1: Design Space

Microbenchmark over maintained derived-state operations. This notebook wraps `htap_wkld` and compares `SNAP`, `IVMH`, `MONO`, `DUAL`, and `EPOCH` across three workload mixes: read-heavy, balanced, and write-heavy.

The figure is a 3-panel grouped stacked bar chart using the repo's existing paper style.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import TOL, apply_paper_style, ensure_dirs, run_checked, display_name
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp1_design_space').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}
TXN_NUM = 100
REPEAT = 3

WORKLOADS = {
    'RH': 0.80,
    'B': 0.50,
    'WH': 0.20,
}

BASE_ARGS = [
    '--update-ratio', '0.005',
    '--probe-ratio', '0.0005',
    '--txn-count', '40',
    '--scan-count', '6',
    '--warehouse-count', '5',
    '--scan-reuse-ratio', '0.8',
    '--txn-gc-ratio', '0.05',
]

TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}
TX_ORDER = ['InitLoad', 'Update', 'GbgCollect', 'BuildSnap', 'RecentScan', 'HistoryScan', 'Probe', 'DeltaScan']
COLOR_MAP = {
    'InitLoad': TOL['grey'],
    'Update': TOL['yellow'],
    'GbgCollect': TOL['cyan'],
    'BuildSnap': TOL['purple'],
    'RecentScan': TOL['green'],
    'HistoryScan': TOL['darkgreen'],
    'Probe': TOL['red'],
    'DeltaScan': TOL['blue'],
}
HATCH_MAP = {
    'RecentScan': '///',
    'HistoryScan': '\\\\\\\\',
    'Probe': 'xxx',
    'DeltaScan': '\\\\\\\\',
}
WRITE_OPS = {'InitLoad', 'Update', 'GbgCollect', 'BuildSnap'}
TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
REPAIR_ORDER = {
    'naive': [''],
    'ivmh': [''],
    'heap': ['No Repair', 'Read Repair', 'Write Repair'],
    'chain': ['Write Repair'],
    'par': ['No Repair', 'Read Repair', 'Write Repair'],
}
REPAIR_LABEL = {'': '', 'No Repair': 'NR', 'Read Repair': 'RR', 'Write Repair': 'WR'}

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)

In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')

In [ ]:
def run_workload(name, analytical_ratio):
    csv_path = DATA_DIR / f'sigmod_exp1_{name}.csv'
    if csv_path.exists():
        print(f'Using cached CSV: {csv_path.name}')
        return pd.read_csv(csv_path, keep_default_na=False)

    args = BASE_ARGS + ['--analytical-ratio', str(analytical_ratio)]
    rows = []
    for table_type in TABLE_TYPES:
        print(f'  workload={name} table={table_type}')
        trials = []
        for _ in range(REPEAT):
            result = run_checked([str(BIN), *args, '--table-type', table_type], ROOT, quiet=True)
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {name}/{table_type}')
            trials.append(df)
        df_all = pd.concat(trials, ignore_index=True)
        df_all['tx_type'] = df_all['tx_type'].replace(TX_MAP)
        df_avg = (
            df_all.groupby(['table_type', 'repair_type', 'tx_type'], as_index=False)['duration_ms']
            .mean()
        )
        if table_type in BASELINES:
            df_avg = df_avg[df_avg['repair_type'] == 'Write Repair'].copy()
            df_avg['repair_type'] = ''
        rows.append(df_avg)

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print(f'Saved {csv_path.name}')
    return out

results = {name: run_workload(name, ratio) for name, ratio in WORKLOADS.items()}
summary = []
for name, df in results.items():
    total = df.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].sum()
    total['workload'] = name
    summary.append(total)
display(pd.concat(summary, ignore_index=True))

In [ ]:
def plot_panel(ax, df, title):
    pivot = df.pivot_table(index=['table_type', 'repair_type'], columns='tx_type', values='duration_ms', aggfunc='sum', fill_value=0.0)
    pivot = pivot.reindex(columns=TX_ORDER, fill_value=0.0)

    bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
    x = 0.0
    gap = 0.36
    for table in TABLE_ORDER:
        subs = REPAIR_ORDER[table]
        start = x
        for repair in subs:
            combos.append((table, repair))
            bar_x.append(x)
            minor_labels.append(REPAIR_LABEL[repair])
            x += 0.78
        major_centers.append((start + (x - 0.78)) / 2.0)
        major_labels.append(display_name(table, ''))
        x += gap

    for idx, (table, repair) in enumerate(combos):
        if (table, repair) in pivot.index:
            row = pivot.loc[(table, repair)]
        else:
            row = pd.Series(0.0, index=TX_ORDER)
        bottom = 0.0
        for tx in TX_ORDER:
            value = float(row.get(tx, 0.0)) / TXN_NUM
            if value <= 0:
                continue
            if tx in WRITE_OPS:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
            else:
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='white', edgecolor='black', linewidth=0.4)
                ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='none', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
            bottom += value

    ax.set_title(title)
    ax.set_xticks([])
    ax.set_ylabel('Duration (ms / tx)')
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    for xi, label in zip(bar_x, minor_labels):
        ax.text(xi, -0.06, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=8, clip_on=False)
    for xc, label in zip(major_centers, major_labels):
        ax.text(xc, -0.14, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=9, clip_on=False)

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.4), sharey=True)
for ax, (name, _) in zip(axes, WORKLOADS.items()):
    plot_panel(ax, results[name], name)

legend_handles = []
legend_labels = []
for tx in TX_ORDER[::-1]:
    if tx in WRITE_OPS:
        patch = Patch(facecolor=COLOR_MAP[tx], edgecolor='black', linewidth=0.4)
    else:
        patch = Patch(facecolor='white', edgecolor=COLOR_MAP[tx], linewidth=0.9, hatch=HATCH_MAP.get(tx, '///'))
    legend_handles.append(patch)
    legend_labels.append(tx)

fig.legend(legend_handles, legend_labels, title='Operations', loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.08), framealpha=0.95)
fig.tight_layout()
out_pdf = FIGS_DIR / 'sigmod_exp1_design_space.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)